In [1]:
from pyscf import gto, dft
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path
import numpy as np
import os

ROOT_DIR = os.path.dirname(os.path.abspath("__file__"))

xyz_file = "active_site_202604222145.xyz"

with open(xyz_file, "r") as f:
    lines = f.readlines()

# Remove XYZ header
atom_block = "".join(lines[2:])

# DFT basis & Functional
basis = "lanl2dz"
# basis = "cc-pvdz"
#basis = "def2-svp" # lowest, quick screening
#basis = "def2-TZVP" # Higher level of screening
#basis = "def2-TZVPP" # Most accurate level of all screening
#functional = "B3LYP"
#functional = "PBE0"  # Hybrid functional, good for general use.
functional = "HF"

In [2]:
import numpy as np

def create_water():
    """Return water molecule centered at origin"""
    OH = 0.96
    angle = np.deg2rad(104.5)

    O = np.array([0.0, 0.0, 0.0])
    H1 = np.array([OH, 0.0, 0.0])
    H2 = np.array([
        OH * np.cos(angle),
        OH * np.sin(angle),
        0.0
    ])

    return np.array([O, H1, H2])


def random_rotation_matrix():
    """Generate a random 3D rotation matrix"""
    u1, u2, u3 = np.random.rand(3)

    q1 = np.sqrt(1-u1) * np.sin(2*np.pi*u2)
    q2 = np.sqrt(1-u1) * np.cos(2*np.pi*u2)
    q3 = np.sqrt(u1) * np.sin(2*np.pi*u3)
    q4 = np.sqrt(u1) * np.cos(2*np.pi*u3)

    R = np.array([
        [1 - 2*(q3**2 + q4**2), 2*(q2*q3 - q1*q4), 2*(q2*q4 + q1*q3)],
        [2*(q2*q3 + q1*q4), 1 - 2*(q2**2 + q4**2), 2*(q3*q4 - q1*q2)],
        [2*(q2*q4 - q1*q3), 2*(q3*q4 + q1*q2), 1 - 2*(q2**2 + q3**2)]
    ])

    return R

def random_position_around(center, r_min=2.0, r_max=4.0):
    """Sample random position in spherical shell"""
    r = np.random.uniform(r_min, r_max)
    theta = np.random.uniform(0, np.pi)
    phi = np.random.uniform(0, 2*np.pi)

    x = center[0] + r * np.sin(theta) * np.cos(phi)
    y = center[1] + r * np.sin(theta) * np.sin(phi)
    z = center[2] + r * np.cos(theta)

    return np.array([x, y, z])

def place_water_near_zn(zn_pos):
    water = create_water()

    # rotate
    R = random_rotation_matrix()
    water_rot = water @ R.T

    # translate
    O_target = random_position_around(zn_pos)
    shift = O_target - water_rot[0]

    water_final = water_rot + shift

    return water_final  # shape (3,3)

In [3]:
def build_xyz_string(base_atoms, water_coords):
    """
    base_atoms: list of (element, [x,y,z])
    water_coords: np.array shape (3,3)
    """
    xyz = ""

    for elem, coord in base_atoms:
        xyz += f"{elem} {coord[0]} {coord[1]} {coord[2]}\n"

    # add water
    xyz += f"O {water_coords[0][0]} {water_coords[0][1]} {water_coords[0][2]}\n"
    xyz += f"H {water_coords[1][0]} {water_coords[1][1]} {water_coords[1][2]}\n"
    xyz += f"H {water_coords[2][0]} {water_coords[2][1]} {water_coords[2][2]}\n"

    return xyz

In [4]:
N = 100
results = []

for i in range(N):
    water_coords = place_water_near_zn(zn_position)

    atom_str = build_xyz_string(base_atoms, water_coords)

    # run PySCF
    mol = gto.M(atom=atom_str, basis='def2-svp', unit='Ang')
    mf = dft.UKS(mol)
    mf.xc = 'b3lyp'

    energy = mf.kernel()

    results.append({
        "energy": energy,
        "distance": np.linalg.norm(water_coords[0] - zn_position)
    })

NameError: name 'zn_position' is not defined

In [ ]:
from pyscf import gto, dft
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime


# =========================
# USER SETTINGS
# =========================

XYZ_FILE = "active_site_202604222145.xyz"

N_SAMPLES = 50          # start small first; increase later
BASIS = "lanl2dz"
XC = "HF"

CHARGE = 0              # change if needed
SPIN = 0                # 2S = Nalpha - Nbeta

R_MIN = 2.0             # minimum Zn-O(water) distance in Angstrom
R_MAX = 5.0             # maximum Zn-O(water) distance in Angstrom

OUTPUT_DIR = Path("water_sampling_results")
OUTPUT_DIR.mkdir(exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")


# =========================
# READ BASE XYZ FILE
# =========================

def read_xyz(filename):
    atoms = []

    with open(filename, "r") as f:
        lines = f.readlines()

    for line in lines[2:]:  # skip XYZ header
        parts = line.split()
        if len(parts) == 4:
            elem = parts[0]
            coord = np.array([float(parts[1]), float(parts[2]), float(parts[3])])
            atoms.append((elem, coord))

    return atoms


def find_zn_position(atoms):
    for elem, coord in atoms:
        if elem.lower() == "zn":
            return coord

    raise ValueError("No Zn atom found in the XYZ file.")


# =========================
# WATER GEOMETRY
# =========================

def create_water():
    """
    Creates one water molecule with realistic geometry.
    O-H bond length ~0.96 Angstrom
    H-O-H angle ~104.5 degrees
    """
    OH = 0.96
    angle = np.deg2rad(104.5)

    O = np.array([0.0, 0.0, 0.0])
    H1 = np.array([OH, 0.0, 0.0])
    H2 = np.array([
        OH * np.cos(angle),
        OH * np.sin(angle),
        0.0
    ])

    return np.array([O, H1, H2])


def random_rotation_matrix():
    """
    Generates a random 3D rotation matrix using a random quaternion.
    """
    u1, u2, u3 = np.random.rand(3)

    q1 = np.sqrt(1 - u1) * np.sin(2 * np.pi * u2)
    q2 = np.sqrt(1 - u1) * np.cos(2 * np.pi * u2)
    q3 = np.sqrt(u1) * np.sin(2 * np.pi * u3)
    q4 = np.sqrt(u1) * np.cos(2 * np.pi * u3)

    R = np.array([
        [1 - 2 * (q3**2 + q4**2), 2 * (q2*q3 - q1*q4),     2 * (q2*q4 + q1*q3)],
        [2 * (q2*q3 + q1*q4),     1 - 2 * (q2**2 + q4**2), 2 * (q3*q4 - q1*q2)],
        [2 * (q2*q4 - q1*q3),     2 * (q3*q4 + q1*q2),     1 - 2 * (q2**2 + q3**2)]
    ])

    return R


def random_position_around(center, r_min=2.0, r_max=5.0):
    """
    Samples a random point in a spherical shell around Zn.
    """
    r = np.random.uniform(r_min, r_max)
    theta = np.random.uniform(0, np.pi)
    phi = np.random.uniform(0, 2 * np.pi)

    x = center[0] + r * np.sin(theta) * np.cos(phi)
    y = center[1] + r * np.sin(theta) * np.sin(phi)
    z = center[2] + r * np.cos(theta)

    return np.array([x, y, z])


def place_water_near_zn(zn_position, r_min=2.0, r_max=5.0):
    """
    Places one randomly oriented water molecule near Zn.
    The oxygen atom is placed at a sampled distance from Zn.
    """
    water = create_water()

    R = random_rotation_matrix()
    water_rotated = water @ R.T

    oxygen_target = random_position_around(
        zn_position,
        r_min=r_min,
        r_max=r_max
    )

    shift = oxygen_target - water_rotated[0]
    water_final = water_rotated + shift

    return water_final


# =========================
# BUILD PYSCF GEOMETRY
# =========================

def build_atom_string(base_atoms, water_coords):
    atom_string = ""

    for elem, coord in base_atoms:
        atom_string += f"{elem} {coord[0]:.8f} {coord[1]:.8f} {coord[2]:.8f}\n"

    atom_string += f"O {water_coords[0,0]:.8f} {water_coords[0,1]:.8f} {water_coords[0,2]:.8f}\n"
    atom_string += f"H {water_coords[1,0]:.8f} {water_coords[1,1]:.8f} {water_coords[1,2]:.8f}\n"
    atom_string += f"H {water_coords[2,0]:.8f} {water_coords[2,1]:.8f} {water_coords[2,2]:.8f}\n"

    return atom_string


def save_xyz(filename, base_atoms, water_coords):
    total_atoms = len(base_atoms) + 3

    with open(filename, "w") as f:
        f.write(f"{total_atoms}\n")
        f.write("Active site with randomly sampled water molecule\n")

        for elem, coord in base_atoms:
            f.write(f"{elem} {coord[0]:.8f} {coord[1]:.8f} {coord[2]:.8f}\n")

        f.write(f"O {water_coords[0,0]:.8f} {water_coords[0,1]:.8f} {water_coords[0,2]:.8f}\n")
        f.write(f"H {water_coords[1,0]:.8f} {water_coords[1,1]:.8f} {water_coords[1,2]:.8f}\n")
        f.write(f"H {water_coords[2,0]:.8f} {water_coords[2,1]:.8f} {water_coords[2,2]:.8f}\n")


# =========================
# RUN PYSCF
# =========================

def run_pyscf(atom_string):
    mol = gto.M(
        atom=atom_string,
        basis=BASIS,
        unit="Ang",
        charge=CHARGE,
        spin=SPIN
    )

    mf = dft.UKS(mol)
    mf.xc = XC

    energy = mf.kernel()

    mo_energy = mf.mo_energy

    # For UKS, mo_energy has alpha and beta arrays.
    alpha_energies = mo_energy[0]
    alpha_occ = mf.mo_occ[0]

    occupied = alpha_energies[alpha_occ > 0]
    virtual = alpha_energies[alpha_occ == 0]

    homo = occupied[-1]
    lumo = virtual[0]
    gap = lumo - homo

    return energy, homo, lumo, gap, mf.converged


# =========================
# MAIN SAMPLING LOOP
# =========================

base_atoms = read_xyz(XYZ_FILE)
zn_position = find_zn_position(base_atoms)

print("Zn position:", zn_position)

results = []

xyz_output_dir = OUTPUT_DIR / f"xyz_samples_{timestamp}"
xyz_output_dir.mkdir(exist_ok=True)

for i in range(N_SAMPLES):
    print(f"\nRunning sample {i + 1}/{N_SAMPLES}")

    water_coords = place_water_near_zn(
        zn_position,
        r_min=R_MIN,
        r_max=R_MAX
    )

    zn_o_distance = np.linalg.norm(water_coords[0] - zn_position)

    atom_string = build_atom_string(base_atoms, water_coords)

    sample_xyz_file = xyz_output_dir / f"sample_{i:04d}.xyz"
    save_xyz(sample_xyz_file, base_atoms, water_coords)

    try:
        energy, homo, lumo, gap, converged = run_pyscf(atom_string)

        results.append({
            "sample": i,
            "zn_o_water_distance_A": zn_o_distance,
            "total_energy_hartree": energy,
            "homo_hartree": homo,
            "lumo_hartree": lumo,
            "gap_hartree": gap,
            "converged": converged,
            "xyz_file": str(sample_xyz_file)
        })

        print(f"Energy: {energy:.8f} Ha")
        print(f"Zn-O distance: {zn_o_distance:.3f} A")
        print(f"Gap: {gap:.6f} Ha")
        print(f"Converged: {converged}")

    except Exception as e:
        print(f"Sample {i} failed: {e}")

        results.append({
            "sample": i,
            "zn_o_water_distance_A": zn_o_distance,
            "total_energy_hartree": np.nan,
            "homo_hartree": np.nan,
            "lumo_hartree": np.nan,
            "gap_hartree": np.nan,
            "converged": False,
            "xyz_file": str(sample_xyz_file)
        })


# =========================
# SAVE RESULTS
# =========================

df = pd.DataFrame(results)

csv_file = OUTPUT_DIR / f"water_sampling_results_{timestamp}.csv"
df.to_csv(csv_file, index=False)

print("\nDone!")
print(f"Results saved to: {csv_file}")
print(f"XYZ samples saved to: {xyz_output_dir}")

Zn position: [-6.432 -1.533 15.402]

Running sample 1/50
SCF not converged.
SCF energy = -1304.37393486045 after 50 cycles  <S^2> = 2.481622  2S+1 = 3.3055239
Energy: -1304.37393486 Ha
Zn-O distance: 3.997 A
Gap: 0.287215 Ha
Converged: False

Running sample 2/50
